### Text Splitters for chunking documents

- Text splitters break large docs into smaller chunks that will be retrevable individually and fit within model context window limit.


##### Why we need Text Splitter?
- suppose we have loaded a PDF containing 300 pages.
```
Employee Handbook.pdf
↓
Document Loader
↓
1 Document with 100,000 characters
```
- Now user asks : ```what is the maternity leave policy```
- sending entire 300 page document to the LLM is :
    - Expensive
    - slow
    - Exceeds context window
    - Retrivetes irrelevant information
- Becuase of the above reason , we divide the document into smaller chunks. after this retrival becomes efficiant.
```
Large Document
↓
Chunk 1
Chunk 2
Chunk 3
Chunk 4
...
Chunk N
```

##### Where text Spiltters fit in RAG.
PDF
 ↓
Document Loader
 ↓
Document Objects
 ↓
Text Splitter
 ↓
Chunks
 ↓
Embeddings
 ↓
Vector Database
 ↓
Retriever
 ↓
LLM

##### What does a Text Splitter Do?
- input 
```Apache Spark is a distributed processing engine.
It supports batch processing.
It also supports streaming.```

- ouput ```*chunk 1* Apache Spark is a distributed processing engine```
```chunk 2 It supports batch processing```
```chunk 3 It also supports streaming.```


#### Why chunking Matters ?
- LLMs have a limited context window. You can't stuff an entire 100-page PDF into one prompt. So you split documents into smaller chunks, embed each chunk, and retrieve only the most relevant ones at query time. But chunking is not trivial — split too small and you lose context; split too large and you waste tokens and hurt retrieval precision.

- Two parameters control every splitter:

    1. chunk_size — maximum size of each chunk (in characters or tokens)
    2. chunk_overlap — how many characters overlap between consecutive chunks (prevents context from being cut off at boundaries)

#### Level 1 — CharacterTextSplitter (simplest)
- Splits on a single character (default "\n\n") and counts by character length.
- it splits based on seprator 

In [ ]:
from langchain_text_splitters import CharacterTextSplitter
text = """LangChain is a framework for building LLM applications.

It provides tools for chaining prompts together.
It also supports memory, agents, and retrieval.
"""
spilitter = CharacterTextSplitter(
    separator = "\n\n", # this split on double new line (paragraph boundary)
    chunk_size = 100,
    chunk_overlap = 20,
    length_function = len  # this will count by characters 
)

chunks = spilitter.split_text(text)

for i, c in enumerate(chunks):
    print(f"----Chunk {i+1}---")
    print(c)
# chunk_number = 1
# for c in chunks:
#     print("---chunk---",chunk_number,"----")
#     print(c)
#     chunk_number = chunk_number + 1 # it increase the count by 1

# it works fine for plain text. Fails when your separator doesnot exist or paragraphs are very long.


----Chunk 1---
LangChain is a framework for building LLM applications.
----Chunk 2---
It provides tools for chaining prompts together.
It also supports memory, agents, and retrieval.


- There is a problem with character splitter. suppose we have ```Apache Spark is a distributed processing framework used for large-scale data processing.``` if it splits occures at 50 characters.
- chunk 1 : ```Apache Spark is a distributed processing frame```
- chunk 2 : ```work used for large-scale data processing.```
- here the main thing is sentence is broken and we miss the context for the complete sentence it will become the individual sentences so that is where we will use the RecursiveCharacterTextSplitter.

#### Level 2 — RecursiveCharacterTextSplitter (the default workhorse)
- This is what we should use 90% of the time. It tries a list of separators in order — ["\n\n", "\n", " ", ""] — falling back to the next one only if the chunk is still too large. This respects natural language structure.
- The RecursiveCharacterTextSplitter is a smart text-splitting tool in the LangChain framework designed for Retrieval-Augmented Generation (RAG) and Large Language Model (LLM) workflows. It breaks large documents into manageable chunks by prioritizing natural language structures (like paragraphs and sentences) before enforcing size limits.
- How It Works:
- Instead of brutally slicing text strictly by character count, it recursively uses a list of separators (by default: ["\n\n", "\n", " ", ""]) in the following order:
    1. Paragraphs (\n\n): Tries to split the text into logical paragraphs.
    2. Lines (\n): If paragraphs are too large, it falls back to splitting by lines.
    3. Words ( ): If lines are still too large, it splits by spaces to keep words intact.
    4. Characters (""): As a last resort, it will break at the character level.

##### Key parameters 
1. chunk_size: The maximum number of characters allowed in a single chunk (default is usually 4,000).
2. chunk_overlap :  The number of characters that the previous and current chunk should share. This helps maintain context across chunk boundaries.
3. separators: An optional customized list of characters or strings that you want the splitter to look for, ordered by priority.

In [16]:
# python Example
from langchain_text_splitters import RecursiveCharacterTextSplitter

# initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100,
    chunk_overlap = 20,
    separators = ["\n\n","\n","",""]
)

# create the text to see the example
text = "LangChain is a framework for developing applications powered by language models. It enables applications that are context-aware and reasoned."
# generate the chunks
chunks = text_splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f"chunk {i+1}:{chunk}")


chunk 1:LangChain is a framework for developing applications powered by language models. It enables applicat
chunk 2:It enables applications that are context-aware and reasoned.


- if we observe on the above response the chunk over lap we can see ``` it enables applications``` part , here we are not lost the context of chunk 1 , some characters are carried for chunk 2 also to maintain the context and continuity.

- Blindly splitting text at exact character counts frequently breaks words and sentences in half, which degrades the quality of vector embeddings and retrieval results. RecursiveCharacterTextSplitter protects semantic meaning by grouping related concepts together and only breaking a chunk further down the list if it still exceeds your size limit.

In [19]:
# example 2 :
from langchain_text_splitters import RecursiveCharacterTextSplitter

long_text = """
Chapter 1: Introduction to LangChain

LangChain is an open-source framework designed to simplify the creation of 
applications using large language models. It was created by Harrison Chase in 2022.

The framework provides a standard interface for chains, lots of integrations 
with other tools, and end-to-end chains for common applications.

Chapter 2: Core Concepts

The core abstraction in LangChain is the "chain" — a sequence of calls to 
components, which can include other chains. This makes it highly composable.

Memory allows chains to remember previous interactions. Without memory, every 
call to a chain is stateless and independent.
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]  # tries these in order
)

chunks = splitter.split_text(long_text)
for i, c in enumerate(chunks):
    print(f"Chunk {i+1} [{len(c)} chars]: {c[:80]}...")

Chunk 1 [197 chars]: Chapter 1: Introduction to LangChain

LangChain is an open-source framework desi...
Chunk 2 [168 chars]: The framework provides a standard interface for chains, lots of integrations 
wi...
Chunk 3 [177 chars]: Chapter 2: Core Concepts

The core abstraction in LangChain is the "chain" — a s...
Chunk 4 [124 chars]: Memory allows chains to remember previous interactions. Without memory, every 
c...


#### Why Recursive method is used mostly.
- it tries multiple separators:
1. paragraphs (\n\n)
2. Lines (\n)
3. Spaces (" ")
4. Characters (if nothing above works)

- visusal flow 
```
Paragraph
↓
Sentence
↓
Word
↓
Character
```

#### Level 3 — Token-based splitting (for LLM token limits)
- Token-based splitting in LangChain divides a large text into smaller chunks based on the number of tokens instead of characters. Large Language Models (LLMs) calculate context windows, API costs, and processing thresholds entirely using tokens, making this method essential for accurate memory and context management.
- Character count ≠ token count. GPT-4 counts "LangChain" as 2 tokens, not 9 characters. When you need to fit within a strict token budget, use a token splitter.



In [20]:
from langchain_text_splitters import TokenTextSplitter

splitter = TokenTextSplitter(
    chunk_size = 256, # in tokens not characters
    chunk_overlap = 32
)

chunks = splitter.split_text(long_text)
print(f"Number of Chunks:{len(chunks)}")

Number of Chunks:60


In [30]:

# Use the tok-token based version for open AI models.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# use the tiktoken under the hood 
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name = "cl100k_base",
    chunk_size = 512,
    chunk_overlap = 64
)
raw_text = "LangChain makes building LLM applications easy. It has text splitters."
chunks = splitter.split_text(raw_text)
print(f"Number of chunks:{len(chunks)}")

Number of chunks:1


This is important when LLM charges per token or has hard context limits. If you embed with text-embedding-3-small (max 8191 tokens), splitting by character can accidentally create chunks that exceed the limit.

#### Level 4 — Structure-aware splitters


#### Markdown
- respects heading hierarchy (#,##,###) splitting. keeps related content together.

In [31]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
markdown_doc = """
# Annual Report 2024

## Executive Summary
Revenue grew by 23% year-over-year driven by expansion in South Asia.

## Financial Highlights
Total revenue: $4.5B
Operating margin: 18%

### Q4 Performance
Q4 showed the strongest growth at 31% YoY.

## Risk Factors
Market volatility remains a concern heading into 2025.
"""
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on = [
        ('#','h1'),
        ('##','h2'),
        ('###','h3')
    ]
)
chunks = splitter.split_text(markdown_doc)

for c in chunks:
    print(c.page_content[:60])
    print(c.metadata) # heading context stored in metadata
    print("-----")

Revenue grew by 23% year-over-year driven by expansion in So
{'h1': 'Annual Report 2024', 'h2': 'Executive Summary'}
-----
Total revenue: $4.5B
Operating margin: 18%
{'h1': 'Annual Report 2024', 'h2': 'Financial Highlights'}
-----
Q4 showed the strongest growth at 31% YoY.
{'h1': 'Annual Report 2024', 'h2': 'Financial Highlights', 'h3': 'Q4 Performance'}
-----
Market volatility remains a concern heading into 2025.
{'h1': 'Annual Report 2024', 'h2': 'Risk Factors'}
-----


HTML

In [33]:
from langchain_text_splitters import HTMLHeaderTextSplitter

html = """
<html>
<body>
  <h1>Product Manual</h1>
  <h2>Installation</h2>
  <p>Download the installer from our website and run setup.exe.</p>
  <h2>Configuration</h2>
  <p>Open config.yml and set your API key in the credentials section.</p>
</body>
</html>
"""

splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=[("h1", "title"), ("h2", "section")]
)

chunks = splitter.split_text(html)
for c in chunks:
    print(c.metadata)
    print(c.page_content)
    print("---")

{'title': 'Product Manual'}
Product Manual
---
{'title': 'Product Manual', 'section': 'Installation'}
Installation
---
{'title': 'Product Manual', 'section': 'Installation'}
Download the installer from our website and run setup.exe.
---
{'title': 'Product Manual', 'section': 'Configuration'}
Configuration
---
{'title': 'Product Manual', 'section': 'Configuration'}
Open config.yml and set your API key in the credentials section.
---


#### Level 5 — Semantic Chunking (advanced)
- instead of splitting by character count or structure, semantic chunking embeds each sentence and splits where the embedding similarity drops , that is where the topic already chnages 

In [38]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",  # or "standard_deviation", "interquartile"
    breakpoint_threshold_amount=95           # split at 95th percentile of distance jumps
)

text = """
LangChain is a framework for building LLM applications. It provides 
chains, agents, and memory. The library is written in Python and JavaScript.

The stock market closed higher today, with the S&P 500 gaining 1.2%. 
Tech stocks led the rally. Apple gained 3% on strong earnings guidance.

Neural networks are composed of layers of interconnected nodes. 
Backpropagation adjusts weights to minimize the loss function during training.
"""

chunks = splitter.split_text(text)
for i, c in enumerate(chunks):
    print(f"Chunk {i+1}: {c[:80]}")

C:\Users\subramani.v\AppData\Local\Temp\ipykernel_23912\1925032555.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

- The splitter detects that the first paragraph is about LangChain/AI, the second is about stocks, and the third is about neural networks — and splits at those semantic boundaries instead of arbitrary character counts.
- This is the most intelligent splitter but also the slowest and most expensive (needs embedding calls during ingestion).



In [ ]:
## Full production Pipeline
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Load
loader = DirectoryLoader("docs/", glob="**/*.pdf", loader_cls=PyPDFLoader)
raw_docs = loader.load()
print(f"Loaded: {len(raw_docs)} pages")

# 2. Split
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True   # adds 'start_index' to metadata — useful for debugging
)
chunks = splitter.split_documents(raw_docs)
print(f"Chunks: {len(chunks)}")

# 3. Inspect before embedding (always do this)
print(chunks[0].page_content[:300])
print(chunks[0].metadata)
# {'source': 'docs/report.pdf', 'page': 0, 'start_index': 0}

# 4. Embed and store
vectorstore = Chroma.from_documents(
    chunks,
    OpenAIEmbeddings(),
    persist_directory="./chroma_db"
)
print("Done.")

ModuleNotFoundError: No module named 'LangChain'

In [45]:
## use hugging face for free embedding models 
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Initialize local HuggingFace Embeddings (runs on your CPU/GPU)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. Build Chroma Vector Store
vectorstore = Chroma.from_documents(
    chunks,
    embeddings,
    persist_directory="./chroma_db"
)
print("Done.")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3227.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Done.


Text Splitters in LangChain divide large documents into smaller chunks so they can be efficiently embedded, stored, and retrieved. The most commonly used splitter is RecursiveCharacterTextSplitter, which recursively splits text using multiple separators while preserving context through chunk overlap. Advanced chunking strategies include token-based, semantic, markdown, HTML, and parent-child chunking